# An Interpretable LLM-Based Classifier for Research Articles on Mathematics Teaching Specialized Knowledge (MTSK)

**Journal:** Applied Artificial Intelligence — Taylor & Francis  
**Authors:** Carolina Rojas Celis · Luis Miguel Elorreaga  
**Institution:** Universidad El Bosque — Maestría en Estadística Aplicada y Ciencia de Datos  

---

## Experiment 13 — Fixed seed SEED = 7

This notebook is one of three independent runs (SEED = 3, 5, 7) used to compute  
mean ± standard deviation of validation metrics reported in the article.

### Model architecture
- **Backbone:** `intfloat/multilingual-e5-large` (frozen during fine-tuning)
- **Head:** Dropout(0.3) + Linear(1024 → 5)
- **Input:** concatenation of title + abstract + keywords with prefix `query:`

### Thematic categories (labels)
| Label | Description |
|-------|-------------|
| T1 | Initial teacher training |
| T2 | Training of teacher educators |
| T3 | MTSK in different mathematical topics and levels |
| T4 | Development of MTSK |
| T5 | Extensions of the MTSK framework |

### Notebook structure
1. Installation & imports  
2. Reproducibility seed  
3. Data loading  
4. Preprocessing & tokenization  
5. Model definition  
6. Training with early stopping  
7. Evaluation: metrics, confusion matrix, per-class report  
8. Interpretability: SHAP global + per-class + per-document  
9. Export to Hugging Face Hub  

### Data availability
The dataset (`DATACEROMTSK_AUMENTADO.xlsx`) is **not included** in this repository  
due to copyright restrictions on article abstracts. It is available upon  
reasonable request to the corresponding author.

---
## 1 · Installation

In [ ]:
# Install all dependencies
# shap is installed separately because it requires a specific build step
!pip install -q transformers datasets torch huggingface_hub evaluate \
             scikit-learn matplotlib seaborn openpyxl
!pip install -q shap
print("Installation complete ✅")

---
## 2 · Imports & reproducibility seed

In [ ]:
import os
import gc
import random
import pickle
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from transformers import (
    AutoTokenizer,
    AutoModel,
    get_linear_schedule_with_warmup,
)
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix,
)
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import shap

# ── Reproducibility: same seed for Python, NumPy, and PyTorch ──────────────
# NOTE: SEED controls model initialisation and training shuffle.
# The train/val split uses the SAME seed for consistency across runs.
SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
# Makes CUDA operations deterministic (slight speed trade-off)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"Seed fixed: SEED={SEED} ✅")
print(f"PyTorch version: {torch.__version__}")

---
## 3 · Data loading

**Expected file:** `DATACEROMTSK_AUMENTADO.xlsx`  
Place it in the same folder as this notebook, or adjust `DATA_PATH` below.  
On Google Colab, mount your Drive and set the path accordingly.

Required columns:
- `Título` — article title
- `Resumen (texto literal)` — abstract
- `Palabras clave (texto literal)` — keywords
- `Clasificacion tematica` — thematic label (T1–T5)

In [ ]:
# ── Option A: Google Colab (mount Drive) ───────────────────────────────────
# Uncomment the two lines below if running on Colab:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# ── DATA_PATH: adjust to your environment ──────────────────────────────────
# Option A (Colab + Drive):  "/content/drive/MyDrive/TG Maestria/DATACEROMTSK_AUMENTADO.xlsx"
# Option B (local / same folder): "data/DATACEROMTSK_AUMENTADO.xlsx"
DATA_PATH = "data/DATACEROMTSK_AUMENTADO.xlsx"  # ← change if needed

df = pd.read_excel(DATA_PATH)

print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nLabel distribution:")
print(df["Clasificacion tematica"].value_counts().sort_index())
df.head(3)

---
## 4 · Preprocessing, label encoding & train/val split

In [ ]:
# ── Build input texts ─────────────────────────────────────────────────────
# multilingual-e5 requires the prefix "query:" for classification inputs.
# We concatenate title + abstract + keywords, replacing NaN with empty string.
comments = (
    "query: "
    + df["Título"].fillna("").astype(str)
    + " "
    + df["Resumen (texto literal)"].fillna("").astype(str)
    + " "
    + df["Palabras clave (texto literal)"].fillna("").astype(str)
).str.strip().tolist()

# ── Label encoding ────────────────────────────────────────────────────────
labels_raw    = df["Clasificacion tematica"].tolist()
nam_labels    = sorted(df["Clasificacion tematica"].unique().tolist())
num_labels    = len(nam_labels)
label2idx     = {label: idx for idx, label in enumerate(nam_labels)}
idx2label     = {idx: label for idx, label in enumerate(nam_labels)}
labels        = [label2idx[l] for l in labels_raw]

# ── Validation ────────────────────────────────────────────────────────────
bad = [i for i, c in enumerate(comments) if not isinstance(c, str)]
assert len(bad) == 0, f"Non-string entries found at indices: {bad}"
print(f"Total samples  : {len(comments)}")
print(f"Classes ({num_labels})    : {nam_labels}")
print(f"label → index  : {label2idx}")

# ── Stratified train / val split (80 / 20) ────────────────────────────────
# IMPORTANT: random_state=SEED (same as global seed) so the split is
# reproducible AND varies across runs (SEED=3, 5, 7), enabling
# a valid mean ± std computation across independent runs.
train_comments, val_comments, train_labels, val_labels = train_test_split(
    comments, labels,
    test_size=0.2,
    random_state=SEED,   # ← was hardcoded to 42; now uses SEED
    stratify=labels
)
print(f"\nTrain size: {len(train_comments)} | Val size: {len(val_comments)}")

---
## 5 · Tokenisation & Dataset

In [ ]:
MODEL_NAME = "intfloat/multilingual-e5-large"

tokenizer       = AutoTokenizer.from_pretrained(MODEL_NAME)
train_encodings = tokenizer(train_comments, truncation=True, padding=True, max_length=128)
val_encodings   = tokenizer(val_comments,   truncation=True, padding=True, max_length=128)
print(f"Tokenisation done. Max length: 128 tokens ✅")


class MTSKDataset(torch.utils.data.Dataset):
    """PyTorch Dataset wrapping tokenised encodings and integer labels."""
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels    = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

---
## 6 · Model definition

In [ ]:
# ── Free GPU memory from previous runs ────────────────────────────────────
for obj_name in ["fine_tuned_model", "base_model"]:
    if obj_name in dir():
        del globals()[obj_name]
gc.collect()
torch.cuda.empty_cache()

# ── Datasets & loaders ────────────────────────────────────────────────────
train_dataset = MTSKDataset(train_encodings, train_labels)
val_dataset   = MTSKDataset(val_encodings,   val_labels)
train_loader  = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader    = DataLoader(val_dataset,   batch_size=16, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


class MTSKClassifier(torch.nn.Module):
    """
    Fine-tuned classifier on top of multilingual-e5-large.

    Architecture:
        backbone  : intfloat/multilingual-e5-large  (hidden_size = 1024)
        pooling   : [CLS] token (first token of last hidden state)
        head      : Dropout(0.3) → Linear(1024, num_labels)

    The backbone weights are updated during fine-tuning (no freezing),
    allowing the model to adapt its representations to the MTSK domain.
    """
    def __init__(self, backbone, num_labels):
        super().__init__()
        self.backbone   = backbone
        self.dropout    = torch.nn.Dropout(0.3)
        self.classifier = torch.nn.Linear(backbone.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        out        = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        pooled     = out.last_hidden_state[:, 0, :]   # [CLS] pooling
        pooled     = self.dropout(pooled)
        logits     = self.classifier(pooled)
        return logits


base_model       = AutoModel.from_pretrained(MODEL_NAME)
fine_tuned_model = MTSKClassifier(base_model, num_labels).to(device)
total_params     = sum(p.numel() for p in fine_tuned_model.parameters())
trainable_params = sum(p.numel() for p in fine_tuned_model.parameters() if p.requires_grad)
print(f"Model loaded: {MODEL_NAME}")
print(f"Total params   : {total_params:,}")
print(f"Trainable params: {trainable_params:,}")

---
## 7 · Training with weighted loss & early stopping

In [ ]:
# ── Class weights (handle imbalance) ─────────────────────────────────────
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(num_labels),
    y=train_labels,
)
weight_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
print("Class weights:", dict(zip(nam_labels, class_weights.round(3))))

# ── Optimizer & scheduler ─────────────────────────────────────────────────
NUM_EPOCHS    = 20
PATIENCE      = 3
optimizer     = AdamW(fine_tuned_model.parameters(), lr=5e-5)
total_steps   = len(train_loader) * NUM_EPOCHS
scheduler     = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps  = total_steps // 10,
    num_training_steps= total_steps,
)

# ── Training loop ─────────────────────────────────────────────────────────
best_f1           = 0.0
best_preds        = []
best_true         = []
epochs_no_improve = 0
history = {"epoch": [], "train_loss": [], "val_f1": [], "val_acc": [],
           "val_precision": [], "val_recall": []}

for epoch in range(NUM_EPOCHS):
    # ── Train ──
    fine_tuned_model.train()
    total_loss = 0.0
    for batch in train_loader:
        optimizer.zero_grad()
        logits = fine_tuned_model(
            batch["input_ids"].to(device),
            batch["attention_mask"].to(device),
        )
        loss = torch.nn.functional.cross_entropy(
            logits, batch["labels"].to(device), weight=weight_tensor
        )
        loss.backward()
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)

    # ── Validate ──
    fine_tuned_model.eval()
    preds_ep, true_ep = [], []
    for batch in val_loader:
        with torch.no_grad():
            out = fine_tuned_model(
                batch["input_ids"].to(device),
                batch["attention_mask"].to(device),
            )
        preds_ep.extend(torch.argmax(out, dim=-1).cpu().numpy())
        true_ep.extend(batch["labels"].cpu().numpy())

    acc  = accuracy_score(true_ep,  preds_ep)
    f1   = f1_score(true_ep,        preds_ep, average="macro", zero_division=0)
    prec = precision_score(true_ep, preds_ep, average="macro", zero_division=0)
    rec  = recall_score(true_ep,    preds_ep, average="macro", zero_division=0)

    history["epoch"].append(epoch + 1)
    history["train_loss"].append(round(avg_loss, 4))
    history["val_f1"].append(round(f1, 4))
    history["val_acc"].append(round(acc, 4))
    history["val_precision"].append(round(prec, 4))
    history["val_recall"].append(round(rec, 4))

    print(f"Epoch {epoch+1:02d}/{NUM_EPOCHS} | Loss: {avg_loss:.4f} "
          f"| Acc: {acc:.4f} | F1: {f1:.4f} | Prec: {prec:.4f} | Rec: {rec:.4f}")

    if f1 > best_f1:
        best_f1, best_preds, best_true = f1, preds_ep.copy(), true_ep.copy()
        epochs_no_improve = 0
        torch.save(fine_tuned_model.state_dict(), "best_model.pt")
        print(f"  ✅ New best F1 = {best_f1:.4f}  (model saved)")
    else:
        epochs_no_improve += 1
        print(f"  ⏳ No improvement ({epochs_no_improve}/{PATIENCE})")
        if epochs_no_improve >= PATIENCE:
            print(f"  ⏹  Early stopping at epoch {epoch+1}")
            break

print(f"\n══ BEST MACRO F1: {best_f1:.4f} ══")

---
## 8 · Evaluation
### 8.1 · Training curves (loss + 4 metrics)

In [ ]:
best_epoch = history["val_f1"].index(max(history["val_f1"])) + 1
epochs_run = history["epoch"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Left: Training loss ───────────────────────────────────────────────────
ax = axes[0]
ax.plot(epochs_run, history["train_loss"],
        color="#E74C3C", lw=2, marker="o", ms=5, label="Train loss")
ax.axvline(best_epoch, color="#2ECC71", ls="--", lw=1.5,
           label=f"Best epoch ({best_epoch})")
ax.set_title("Training Loss", fontsize=13, fontweight="bold")
ax.set_xlabel("Epoch")
ax.set_ylabel("Cross-entropy loss (weighted)")
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
ax.legend()
ax.grid(alpha=0.3)

# ── Right: 4 validation metrics ──────────────────────────────────────────
ax = axes[1]
ax.plot(epochs_run, history["val_f1"],
        color="#3498DB", lw=2, marker="o", ms=5, label="F1 macro")
ax.plot(epochs_run, history["val_acc"],
        color="#F39C12", lw=2, marker="s", ms=5, label="Accuracy")
ax.plot(epochs_run, history["val_precision"],
        color="#9B59B6", lw=2, marker="^", ms=5, label="Precision macro")
ax.plot(epochs_run, history["val_recall"],
        color="#1ABC9C", lw=2, marker="D", ms=5, label="Recall macro")
ax.axvline(best_epoch, color="#2ECC71", ls="--", lw=1.5,
           label=f"Best epoch ({best_epoch})")
ax.axhline(0.70, color="#888", ls=":", lw=1.2, label="Target threshold (0.70)")
ax.set_title("Validation Metrics (macro)", fontsize=13, fontweight="bold")
ax.set_xlabel("Epoch")
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.suptitle(
    f"Experiment 13 (SEED={SEED}) — Best F1={max(history['val_f1']):.4f} at epoch {best_epoch}",
    fontsize=12, fontweight="bold",
)
plt.tight_layout()
os.makedirs("outputs", exist_ok=True)
plt.savefig(f"outputs/exp13_seed{SEED}_training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved → outputs/exp13_seed{}_training_curves.png".format(SEED))

### 8.2 · Per-class classification report

In [ ]:
# Load the best checkpoint
fine_tuned_model.load_state_dict(torch.load("best_model.pt", map_location=device))
fine_tuned_model.eval()

all_preds, all_true = [], []
for batch in val_loader:
    with torch.no_grad():
        out = fine_tuned_model(
            batch["input_ids"].to(device),
            batch["attention_mask"].to(device),
        )
    all_preds.extend(torch.argmax(out, dim=-1).cpu().numpy())
    all_true.extend(batch["labels"].cpu().numpy())

print("=" * 60)
print(f"CLASSIFICATION REPORT — Experiment 13 (SEED={SEED})")
print("=" * 60)
print(classification_report(all_true, all_preds, target_names=nam_labels, digits=4))

# Summary row
print(f"{'Metric':<20} {'Value':>8}")
print("-" * 30)
print(f"{'Accuracy':<20} {accuracy_score(all_true, all_preds):>8.4f}")
print(f"{'F1 macro':<20} {f1_score(all_true, all_preds, average='macro', zero_division=0):>8.4f}")
print(f"{'Precision macro':<20} {precision_score(all_true, all_preds, average='macro', zero_division=0):>8.4f}")
print(f"{'Recall macro':<20} {recall_score(all_true, all_preds, average='macro', zero_division=0):>8.4f}")

### 8.3 · Confusion matrix (counts + normalised)

In [ ]:
cm      = confusion_matrix(all_true, all_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)  # row-normalised

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Left: raw counts ──────────────────────────────────────────────────────
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Blues",
    xticklabels=nam_labels, yticklabels=nam_labels,
    linewidths=0.5, ax=axes[0],
)
axes[0].set_title("Confusion Matrix (counts)", fontsize=13, fontweight="bold")
axes[0].set_ylabel("True label")
axes[0].set_xlabel("Predicted label")

# ── Right: row-normalised (recall per class on diagonal) ─────────────────
sns.heatmap(
    cm_norm, annot=True, fmt=".2f", cmap="Blues",
    xticklabels=nam_labels, yticklabels=nam_labels,
    linewidths=0.5, vmin=0, vmax=1, ax=axes[1],
)
axes[1].set_title("Confusion Matrix (row-normalised)", fontsize=13, fontweight="bold")
axes[1].set_ylabel("True label")
axes[1].set_xlabel("Predicted label")

plt.suptitle(
    f"Experiment 13 (SEED={SEED}) — Best Model (F1={best_f1:.4f})",
    fontsize=12, fontweight="bold",
)
plt.tight_layout()
plt.savefig(f"outputs/exp13_seed{SEED}_confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved → outputs/exp13_seed{}_confusion_matrix.png".format(SEED))

### 8.4 · Per-class metrics bar chart

Precision, Recall and F1 for each thematic category side by side —  
useful to quickly spot which classes the model finds harder.

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

prec_c, rec_c, f1_c, sup_c = precision_recall_fscore_support(
    all_true, all_preds, labels=list(range(num_labels)), zero_division=0
)

x      = np.arange(num_labels)
width  = 0.25

fig, ax = plt.subplots(figsize=(10, 5))
bars_p = ax.bar(x - width, prec_c, width, label="Precision", color="#9B59B6", alpha=0.85)
bars_r = ax.bar(x,         rec_c,  width, label="Recall",    color="#1ABC9C", alpha=0.85)
bars_f = ax.bar(x + width, f1_c,   width, label="F1",        color="#3498DB", alpha=0.85)

# Annotate bar values
for bars in [bars_p, bars_r, bars_f]:
    for bar in bars:
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.01,
            f"{bar.get_height():.2f}",
            ha="center", va="bottom", fontsize=8,
        )

# Support labels on x-axis
ax.set_xticks(x)
ax.set_xticklabels(
    [f"{l}\n(n={s})" for l, s in zip(nam_labels, sup_c)],
    fontsize=10,
)
ax.set_ylim(0, 1.12)
ax.axhline(0.70, color="#888", ls=":", lw=1.2, label="Target (0.70)")
ax.set_title(
    f"Per-class Precision / Recall / F1 — Experiment 13 (SEED={SEED})",
    fontsize=13, fontweight="bold",
)
ax.set_ylabel("Score")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(f"outputs/exp13_seed{SEED}_per_class_metrics.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved → outputs/exp13_seed{}_per_class_metrics.png".format(SEED))

---
## 9 · Interpretability with SHAP

We use **SHAP (SHapley Additive exPlanations)** with a `Text` masker  
to attribute the model's predictions to individual tokens.

Three levels of analysis are provided:
1. **Global importance per class** — which tokens most influence each category on average
2. **Comparative beeswarm** — distribution of token-level SHAP values across all classes
3. **Document-level analysis** — token highlighting for individual prediction cases (correct ✅ and incorrect ❌)

### 9.1 · SHAP wrapper & explainer

In [ ]:
class SHAPWrapper:
    """
    Wraps MTSKClassifier to return class probabilities (numpy array)
    from raw text strings, as required by shap.Explainer.
    Input : list of strings
    Output: numpy array of shape (n_samples, num_labels) — softmax probabilities
    """
    def __init__(self, model, tokenizer, device):
        self.model     = model
        self.tokenizer = tokenizer
        self.device    = device

    def __call__(self, texts):
        if isinstance(texts, str):
            texts = [texts]
        enc = self.tokenizer(
            list(texts),
            truncation=True, padding=True,
            max_length=128, return_tensors="pt",
        ).to(self.device)
        self.model.eval()
        with torch.no_grad():
            logits = self.model(
                input_ids      = enc["input_ids"],
                attention_mask = enc["attention_mask"],
            )
        return torch.nn.functional.softmax(logits, dim=-1).cpu().numpy()


shap_wrapper = SHAPWrapper(fine_tuned_model, tokenizer, device)
print(f"SHAP wrapper ready ✅")
print(f"Output classes: {list(enumerate(nam_labels))}")

In [ ]:
# ── Build explainer ───────────────────────────────────────────────────────
# We use the full validation set as background reference (up to 50 docs).
# Increasing this improves SHAP stability but raises compute time.
N_SHAP = 20   # number of documents to explain (increase if GPU memory allows)

explainer = shap.Explainer(
    shap_wrapper,
    masker       = shap.maskers.Text(tokenizer),
    output_names = nam_labels,
)
print("Explainer created ✅")
print(f"Explaining {N_SHAP} validation documents...")

shap_values = explainer(val_comments[:N_SHAP])
print(f"SHAP values computed ✅  shape: {shap_values.shape}")
# shape: (N_SHAP, n_tokens, num_labels)

### 9.2 · Global token importance — bar plots per class

Mean absolute SHAP value across all explained documents.  
Tokens at the top are the most globally influential for each thematic category.

In [ ]:
fig, axes = plt.subplots(1, num_labels, figsize=(5 * num_labels, 5), sharey=False)

colors = ["#3498DB", "#E74C3C", "#2ECC71", "#F39C12", "#9B59B6"]

for i, label in enumerate(nam_labels):
    # shap_values[:, :, i] → shape (N_SHAP, n_tokens) for class i
    shap.plots.bar(
        shap_values[:, :, label],
        max_display=12,
        show=False,
        ax=axes[i] if num_labels > 1 else axes,
    )
    ax = axes[i] if num_labels > 1 else axes
    ax.set_title(f"Class {label}", fontsize=11, fontweight="bold", color=colors[i])
    ax.set_xlabel("Mean |SHAP value|")

plt.suptitle(
    f"Global token importance per MTSK category (SEED={SEED})",
    fontsize=13, fontweight="bold", y=1.02,
)
plt.tight_layout()
plt.savefig(
    f"outputs/exp13_seed{SEED}_shap_global_bar.png",
    dpi=150, bbox_inches="tight",
)
plt.show()
print("Figure saved → outputs/exp13_seed{}_shap_global_bar.png".format(SEED))

### 9.3 · Beeswarm — distribution of SHAP values per class

Unlike the bar chart, the beeswarm shows not only the mean importance  
but also the **direction** (positive = pushes toward this class) and the **spread**  
of each token's influence across the explained documents.

In [ ]:
for i, label in enumerate(nam_labels):
    shap.plots.beeswarm(
        shap_values[:, :, label],
        max_display=12,
        show=False,
    )
    plt.title(
        f"SHAP beeswarm — Class {label}",
        fontsize=12, fontweight="bold",
    )
    plt.tight_layout()
    plt.savefig(
        f"outputs/exp13_seed{SEED}_shap_beeswarm_{label}.png",
        dpi=150, bbox_inches="tight",
    )
    plt.show()

print("Beeswarm figures saved ✅")

### 9.4 · Document-level analysis — token highlighting

We select representative documents from the validation set:  
- ✅ **Correct** predictions — confirms which tokens the model correctly relies on  
- ❌ **Incorrect** predictions — reveals the linguistic source of confusion between categories

Red tokens **push toward** the predicted class; blue tokens **push against** it.

In [ ]:
# ── Full prediction table for val[:N_SHAP] ────────────────────────────────
print(f"{'IDX':<5} {'True label':<14} {'Predicted':<14} {'Confidence':>10}  {'Result'}")
print("-" * 55)

correct_idx   = []
incorrect_idx = []

for idx in range(N_SHAP):
    probs      = shap_wrapper([val_comments[idx]])[0]
    pred_idx   = np.argmax(probs)
    true_label = idx2label[val_labels[idx]]
    pred_label = nam_labels[pred_idx]
    conf       = probs[pred_idx]
    result     = "✅" if true_label == pred_label else "❌"

    print(f"{idx:<5} {true_label:<14} {pred_label:<14} {conf:>10.3f}  {result}")

    if true_label == pred_label:
        correct_idx.append(idx)
    else:
        incorrect_idx.append(idx)

print(f"\nCorrect  : {len(correct_idx)} | Incorrect: {len(incorrect_idx)}")

In [ ]:
# ── Select representative cases ───────────────────────────────────────────
# Up to 2 correct + 2 incorrect for qualitative analysis
cases_to_show = correct_idx[:2] + incorrect_idx[:2]

for idx in cases_to_show:
    probs      = shap_wrapper([val_comments[idx]])[0]
    pred_idx   = int(np.argmax(probs))
    true_label = idx2label[val_labels[idx]]
    pred_label = nam_labels[pred_idx]
    result     = "✅ CORRECT" if true_label == pred_label else "❌ INCORRECT"

    print("\n" + "═" * 60)
    print(f"{result}  |  Doc {idx}")
    print(f"  True label   : {true_label}")
    print(f"  Predicted    : {pred_label}  (confidence: {probs[pred_idx]:.3f})")
    print(f"  Text preview : {val_comments[idx][:200]}...")
    print("  Class probability distribution:")
    for lbl, p in zip(nam_labels, probs):
        bar = "█" * int(p * 20)
        print(f"    {lbl}: {bar:<20} {p:.3f}")
    print("═" * 60)

    # Interactive token-level SHAP text plot
    shap.plots.text(shap_values[idx])

### 9.5 · Save SHAP values

In [ ]:
shap_path = f"outputs/exp13_seed{SEED}_shap_values.pkl"
with open(shap_path, "wb") as f:
    pickle.dump(shap_values, f)
print(f"SHAP values saved → {shap_path} ✅")

---
## 10 · Export model to Hugging Face Hub

**Prerequisites (run once):**
1. Create a free account at https://huggingface.co  
2. Go to Settings → Access Tokens → New Token (type **write**)  
3. On Colab: Menu → 🔑 Secrets → Add secret  
   - Name: `huggingface`  
   - Value: `hf_xxxxxxxxxxxx`  

### 10.1 · Login

In [ ]:
from google.colab import userdata
from huggingface_hub import login

login(userdata.get("huggingface"))
print("Login successful ✅")

### 10.2 · Save model artefacts

In [ ]:
# Load best checkpoint
fine_tuned_model.load_state_dict(torch.load("best_model.pt", map_location=device))
fine_tuned_model.eval()

HF_MODEL_DIR = "hf_model"
os.makedirs(HF_MODEL_DIR, exist_ok=True)

# Backbone + tokenizer (standard HF format)
fine_tuned_model.backbone.save_pretrained(HF_MODEL_DIR)
tokenizer.save_pretrained(HF_MODEL_DIR)

# Classification head weights + metadata
torch.save(
    {
        "classifier_state_dict": fine_tuned_model.classifier.state_dict(),
        "label2idx": label2idx,
        "idx2label": idx2label,
        "nam_labels": nam_labels,
        "num_labels": num_labels,
        "model_name": MODEL_NAME,
        "seed": SEED,
    },
    os.path.join(HF_MODEL_DIR, "classifier_head.pt"),
)

print(f"Model artefacts saved to: {HF_MODEL_DIR}")
print(f"Files: {os.listdir(HF_MODEL_DIR)}")

### 10.3 · Upload to Hub

In [ ]:
from huggingface_hub import HfApi, list_repo_files

# ✏️ Replace with your Hugging Face username
HF_REPO = "crojasc2/mtsk-classifier"   # e.g. "your_username/mtsk-classifier"

api = HfApi()
api.create_repo(HF_REPO, exist_ok=True, repo_type="model")
print(f"Repository ready: {HF_REPO}")

api.upload_folder(
    folder_path = HF_MODEL_DIR,
    repo_id     = HF_REPO,
    repo_type   = "model",
)
print(f"\nModel uploaded ✅")
print(f"URL: https://huggingface.co/{HF_REPO}")

print("\nFiles in repository:")
for f in list_repo_files(HF_REPO):
    print(f"  ✅ {f}")

---
## Summary of outputs

All figures are saved in `outputs/` with consistent naming `exp13_seed{SEED}_*`:

| File | Content |
|------|---------|
| `exp13_seed7_training_curves.png` | Loss + Accuracy + F1 + Precision + Recall per epoch |
| `exp13_seed7_confusion_matrix.png` | Counts & row-normalised confusion matrices |
| `exp13_seed7_per_class_metrics.png` | P / R / F1 bar chart per MTSK category |
| `exp13_seed7_shap_global_bar.png` | Mean absolute SHAP per token, all 5 classes |
| `exp13_seed7_shap_beeswarm_T*.png` | Token SHAP distribution per class |
| `exp13_seed7_shap_values.pkl` | Serialised SHAP values for further analysis |

---

## Citation

```bibtex
@article{rojas2025mtsk,
  title     = {An Interpretable {LLM}-Based Classifier for Research Articles
               on Mathematics Teaching Specialized Knowledge ({MTSK})},
  author    = {Rojas Celis, Carolina and Elorreaga, Luis Miguel},
  journal   = {Applied Artificial Intelligence},
  publisher = {Taylor \& Francis},
  year      = {2025},
  note      = {DOI: [to be assigned]}
}
```